In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import numpy as np

# 'market_data.csv' contains historical prices, demand, and other features.

# 1. Load Data
try:
    data = pd.read_csv('market_data.csv')
except FileNotFoundError:
    raise FileNotFoundError("'market_data.csv' not found. Please provide a valid data file.")

# Normalize column names: strip whitespace to avoid KeyError on columns like ' Date'
data.columns = data.columns.str.strip()

# 2. Feature Engineering and Data Preparation
# Validate required columns exist
required_columns = ['Date', 'Price', 'Demand', 'Renewable_Generation']
missing = [col for col in required_columns if col not in data.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}. Found columns: {list(data.columns)}")

# Convert 'Date' column to datetime objects
data['Date'] = pd.to_datetime(data['Date'])
data.set_index('Date', inplace=True)

# Create lagged features for time series
data['Price_lag1'] = data['Price'].shift(1)
data.dropna(inplace=True)

# 3. Define Features and Target
features = ['Price_lag1', 'Demand', 'Renewable_Generation']
target = 'Price'

X = data[features]
y = data[target]

# 4. Split Data for Training and Testing (time-based, no shuffling)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 5. Model Training
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 6. Make Predictions
predictions = model.predict(X_test)

# 7. Evaluation Metrics
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("=== Model Evaluation ===")
print(f"MAE  (Mean Absolute Error):       {mae:.4f}")
print(f"RMSE (Root Mean Squared Error):   {rmse:.4f}")
print(f"R²   (Coefficient of Determination): {r2:.4f}")

# 8. Visualize Results
plt.figure(figsize=(10, 6))
plt.plot(y_test.index, y_test.values, label='Actual Prices')
plt.plot(y_test.index, predictions, label='Predicted Prices', linestyle='--')
plt.title('Energy Price Forecasting')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.tight_layout()
plt.show()